# 10 Post-Apply Validation

Compare what the batch was supposed to do, what the apply log reported, and what the filesystem state shows afterward. Default use is safe for dry-run logs too.

In [1]:
from pathlib import Path
from datetime import datetime
import sys
import pandas as pd


def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / 'src').exists() and (candidate / 'notebooks').exists():
            return candidate
    return start

PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

OUTPUT_DIR = PROJECT_ROOT / 'data' / 'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

from src.reporting import find_latest_output, load_optional_parquet
from src.post_apply_validation import (
    ValidationConfig,
    build_post_apply_validation,
    validation_summary,
)

print('PROJECT_ROOT =', PROJECT_ROOT)
print('OUTPUT_DIR =', OUTPUT_DIR)


PROJECT_ROOT = C:\00_Developement\sch-file-organizer
OUTPUT_DIR = C:\00_Developement\sch-file-organizer\data\outputs


In [2]:
EXECUTABLE_MANIFEST_PATH = find_latest_output(OUTPUT_DIR, 'execution_manifest_ready_')
APPLY_LOG_PATH = find_latest_output(OUTPUT_DIR, 'apply_log_')
POST_INVENTORY_PATH = find_latest_output(OUTPUT_DIR, 'inventory_', exclude_prefixes=['inventory_with_text_'])

SOURCE_BASE_PATH = None  # e.g. r'D:\sandbox_root'
TARGET_BASE_PATH = None  # if None, validator falls back to SOURCE_BASE_PATH
LIVE_FILESYSTEM_CHECK = True
USE_POST_INVENTORY = True

EXPORT_STEM = datetime.now().strftime('%Y%m%d_%H%M%S')

print('EXECUTABLE_MANIFEST_PATH =', EXECUTABLE_MANIFEST_PATH)
print('APPLY_LOG_PATH =', APPLY_LOG_PATH)
print('POST_INVENTORY_PATH =', POST_INVENTORY_PATH)


EXECUTABLE_MANIFEST_PATH = C:\00_Developement\sch-file-organizer\data\outputs\execution_manifest_ready_20260307_102025.parquet
APPLY_LOG_PATH = C:\00_Developement\sch-file-organizer\data\outputs\apply_log_20260307_102045.parquet
POST_INVENTORY_PATH = C:\00_Developement\sch-file-organizer\data\outputs\inventory_smoke_test.parquet


In [3]:
manifest = load_optional_parquet(EXECUTABLE_MANIFEST_PATH)
apply_log = load_optional_parquet(APPLY_LOG_PATH)
post_inventory = load_optional_parquet(POST_INVENTORY_PATH) if USE_POST_INVENTORY else None

print('manifest rows =', 0 if manifest is None else len(manifest))
print('apply_log rows =', 0 if apply_log is None else len(apply_log))
print('post_inventory rows =', 0 if post_inventory is None else len(post_inventory))


manifest rows = 0
apply_log rows = 0
post_inventory rows = 4


In [4]:
config = ValidationConfig(
    source_base_path=SOURCE_BASE_PATH,
    target_base_path=TARGET_BASE_PATH,
    live_filesystem_check=LIVE_FILESYSTEM_CHECK,
    use_post_inventory=USE_POST_INVENTORY,
)

validation = build_post_apply_validation(
    executable_manifest=manifest,
    apply_log=apply_log,
    config=config,
    post_inventory_df=post_inventory,
)

summary = validation_summary(validation)
summary


{'rows': 0, 'validated_moves': 0, 'dry_run_rows': 0, 'manual_review': 0}

In [5]:
def _show(df, cols, n=20):
    safe_cols = [c for c in cols if c in df.columns]
    display(df[safe_cols].head(n))

_show(validation, ['relative_path', 'execution_source_relative_path', 'execution_target_relative_path', 'apply_status', 'validation_status', 'validation_reason'])

display(validation['validation_status'].fillna('missing').value_counts().rename_axis('validation_status').reset_index(name='count'))

review_mask = validation['requires_manual_review'].fillna(False) if 'requires_manual_review' in validation.columns else pd.Series([False] * len(validation), index=validation.index)
_show(validation.loc[review_mask], ['relative_path', 'apply_status', 'validation_status', 'validation_reason'], n=50)


,relative_path,execution_source_relative_path,execution_target_relative_path,apply_status,validation_status,validation_reason


,validation_status,count


,relative_path,apply_status,validation_status,validation_reason


In [6]:
csv_path = OUTPUT_DIR / f'post_apply_validation_{EXPORT_STEM}.csv'
parquet_path = OUTPUT_DIR / f'post_apply_validation_{EXPORT_STEM}.parquet'
validation.to_csv(csv_path, index=False, encoding='utf-8-sig')
validation.to_parquet(parquet_path, index=False)
print(csv_path)
print(parquet_path)


C:\00_Developement\sch-file-organizer\data\outputs\post_apply_validation_20260307_110832.csv
C:\00_Developement\sch-file-organizer\data\outputs\post_apply_validation_20260307_110832.parquet


## How to read the output

- `validated_move`: the apply log reported a move and the current filesystem state matches that expectation.
- `dry_run_not_applied`: the apply log came from a dry-run, so no move should be visible.
- `not_applied_or_needs_review`: the row was blocked, skipped, errored, or otherwise needs inspection.
- `move_not_observed` / `missing_both_after_move`: the apply log said move, but the current filesystem state does not confirm it.